## Simple Integrate&Fire Model

In [1]:
using PlotlyJS

# --- Constants ---
# Membrane properties
C_m = 1.0   # Membrane capacitance (µF/cm^2)
g_L = 0.1   # Leak conductance (mS/cm^2)
E_L = -70.0 # Leak reversal potential (mV)
V_th = -55.0 # Threshold potential (mV)
V_reset = -75.0 # Reset potential (mV)
V_spike = 20 # Spike peak
tau_ref = 2.0 # Refractory period (ms)

# --- Simulation parameters ---
dt = 0.1      # Time step (ms)
t_end = 100.0 # Total simulation time (ms)
t = 0:dt:t_end

# --- Input current ---
I_inj = 2  # Constant injected current (µA/cm^2)

# --- Initialization ---
V = E_L       # Initial membrane potential (mV)
t_last_spike = -tau_ref # Time of last spike (initialized to be before simulation)

# --- Simulation loop ---
spike_times = Float64[] # Store spike times
V_trace = [] # Store membrane potential for plotting
for j in 2:length(t)
    # Update membrane potential if not in refractory period
    if t[j] > t_last_spike + tau_ref
        # Calculate membrane potential change (dV/dt)
        dVdt = (-g_L * (V - E_L) + I_inj) / C_m
        
        # Update membrane potential
        V += dVdt * dt 
    end

    # Check for spike
    if V >= V_th
        push!(spike_times, t[j]) # Record spike time
        V = V_reset # Reset membrane potential
        t_last_spike = t[j] # Update time of last spike
    end
    push!(V_trace, V)
end

# --- Plotting with PlotlyJS ---
trace1 = PlotlyJS.scatter(;x=t, y=V_trace, mode="lines", name="Membrane Potential")
trace2 = PlotlyJS.scatter(;x=spike_times, y=fill(V_th, length(spike_times)), 
                         mode="markers", marker_symbol="x", 
                         marker_size=10, marker_color="red", name="Spikes")

layout = Layout(
    xaxis_title="Time (ms)",
    yaxis_title="Membrane Potential (mV)",
    title="Integrate-and-Fire Neuron Simulation (PlotlyJS)"
)

plt = PlotlyJS.plot([trace1, trace2], layout)
display(plt)

SyntaxError: invalid syntax (1223274014.py, line 1)

## HH_Model

In [ ]:
using PlotlyJS

# Constants
C_m = 1.0   # membrane capacitance (µF/cm^2)
g_Na = 120.0  # maximum sodium conductance (mS/cm^2)
g_K = 36.0   # maximum potassium conductance (mS/cm^2)
g_L = 0.3    # leak conductance (mS/cm^2)
E_Na = 50.0  # sodium reversal potential (mV)
E_K = -77.0  # potassium reversal potential (mV)
E_L = -54.387 # leak reversal potential (mV)

# Time parameters
dt = 0.01   # time step (ms)
t_end = 50.0 # total simulation time (ms)
time = 0:dt:t_end



## First, run simulation without any synaptic input

In [ ]:
 # Initial conditions
 V = -65.0 # Initial membrane potential (mV)
 # m represents the activation of the fast sodium channels. A value of 0 means the channels are fully closed, and 1 means they are fully open.
 # The initial value of m is low because, at rest, most sodium channels are closed. The neuron is not ready to fire an action potential yet.
 m =  0.05 
 # h represents the inactivation of the fast sodium channels. Like m, a value of 0 means full inactivation, and 1 means no inactivation.
 # The initial value of h is relatively high because, at rest, the sodium channels are not inactivated. They are ready to open when the neuron depolarizes.
 h =  0.6
 # n represents the activation of the slower potassium channels. Again, 0 means fully closed, and 1 means fully open.
 # The initial value of n is low because, at the resting membrane potential, most potassium channels are closed.
 n =  0.32

 # Store the results for this synaptic strength
 V_trace = zeros(length(time))
 I_syn_trace = zeros(length(time))

 # Inner simulation loop (time steps)
 for i in 1:length(time)
     
     # Alpha and beta functions for gating variables
     alpha_m = (0.1*(V + 40.0))/(1.0 - exp(-(V + 40.0)/10.0))
     beta_m = 4.0*exp(-(V + 65.0)/18.0)
     alpha_h = 0.07*exp(-(V + 65.0)/20.0)
     beta_h = 1.0/(1.0 + exp(-(V + 35.0)/10.0))
     alpha_n = (0.01*(V + 55.0))/(1.0 - exp(-(V + 55.0)/10.0))
     beta_n = 0.125*exp(-(V + 65.0)/80.0)

     # Update gating variables using forward Euler method
     m += dt * (alpha_m*(1.0 - m) - beta_m*m)
     h += dt * (alpha_h*(1.0 - h) - beta_h*h)
     n += dt * (alpha_n*(1.0 - n) - beta_n*n)

     # Membrane current calculations
     I_Na = g_Na * m^3 * h * (V - E_Na)
     I_K = g_K * n^4 * (V - E_K)
     I_L = g_L * (V - E_L)
     
     # Update membrane potential using forward Euler method
     V += dt * (1/C_m) * (-I_Na - I_K - I_L) 

     # Store the results
     V_trace[i] = V
end

 # Plot V_trace on the first subplot
 plot(time, V_trace, label="V_m", linewidth=2)



## Now add a synaptic input, not large enough to generate a spike

In [ ]:
# Synaptic input parameters
t_start_syn = 10.0  # Synaptic input start time (ms)
t_end_syn = 12.0    # Synaptic input end time (ms)
E_syn = 0.0         # Synaptic reversal potential (mV)

# Create two plots
plt_V = plot()
plt_I = plot()

# Setting a large synaptic input
g_syn_max = 0.02 # Maximum synaptic conductances (mS/cm^2)

# Initial conditions
V = -65.0 # Initial membrane potential (mV)
m = 0.05
h = 0.6
n = 0.32

# Store the results for this synaptic strength
V_trace = zeros(length(time))
I_syn_trace = zeros(length(time))

# Inner simulation loop (time steps)
for i in 1:length(time)
    # Synaptic conductance (simple square pulse)
    g_syn = 0.0
    if t_start_syn <= time[i] <= t_end_syn
        g_syn = g_syn_max
    end

    # Alpha and beta functions for gating variables
    alpha_m = (0.1*(V + 40.0))/(1.0 - exp(-(V + 40.0)/10.0))
    beta_m = 4.0*exp(-(V + 65.0)/18.0)
    alpha_h = 0.07*exp(-(V + 65.0)/20.0)
    beta_h = 1.0/(1.0 + exp(-(V + 35.0)/10.0))
    alpha_n = (0.01*(V + 55.0))/(1.0 - exp(-(V + 55.0)/10.0))
    beta_n = 0.125*exp(-(V + 65.0)/80.0)

    # Update gating variables using forward Euler method
    m += dt * (alpha_m*(1.0 - m) - beta_m*m)
    h += dt * (alpha_h*(1.0 - h) - beta_h*h)
    n += dt * (alpha_n*(1.0 - n) - beta_n*n)

    # Membrane current calculations
    I_Na = g_Na * m^3 * h * (V - E_Na)
    I_K = g_K * n^4 * (V - E_K)
    I_L = g_L * (V - E_L)
    I_syn = g_syn * (V - E_syn)

    # Update membrane potential using forward Euler method
    V += dt * (1/C_m) * (-I_Na - I_K - I_L - I_syn) 

    # Store the results
    V_trace[i] = V
    I_syn_trace[i] = I_syn
end

# Plot V_trace on the first subplot and I_syn_trace on the second
add_trace!(plt_V, scatter(x=time, y=V_trace, mode="lines", name="V_m (g_syn_max = $(g_syn_max))"))
add_trace!(plt_I, scatter(x=time, y=I_syn_trace, mode="lines", name="I_syn (g_syn_max = $(g_syn_max))"))

# Display the plot with both subplots
[plt_V; plt_I]

## Add a large synaptic input!

In [ ]:
# Synaptic input parameters
t_start_syn = 10.0  # Synaptic input start time (ms)
t_end_syn = 12.0    # Synaptic input end time (ms)
E_syn = 0.0         # Synaptic reversal potential (mV)

# Create two plots
plt_V = PlotlyJS.plot()
plt_I = PlotlyJS.plot()

# Setting a large synaptic input
g_syn_max = 0.1 # Maximum synaptic conductances (mS/cm^2)

# Initial conditions
V = -65.0 # Initial membrane potential (mV)
m = 0.05
h = 0.6
n = 0.32

# Store the results for this synaptic strength
V_trace = zeros(length(time))
I_syn_trace = zeros(length(time))

# Inner simulation loop (time steps)
for i in 1:length(time)
    # Synaptic conductance (simple square pulse)
    g_syn = 0.0
    if t_start_syn <= time[i] <= t_end_syn
        g_syn = g_syn_max
    end

    # Alpha and beta functions for gating variables
    alpha_m = (0.1*(V + 40.0))/(1.0 - exp(-(V + 40.0)/10.0))
    beta_m = 4.0*exp(-(V + 65.0)/18.0)
    alpha_h = 0.07*exp(-(V + 65.0)/20.0)
    beta_h = 1.0/(1.0 + exp(-(V + 35.0)/10.0))
    alpha_n = (0.01*(V + 55.0))/(1.0 - exp(-(V + 55.0)/10.0))
    beta_n = 0.125*exp(-(V + 65.0)/80.0)

    # Update gating variables using forward Euler method
    m += dt * (alpha_m*(1.0 - m) - beta_m*m)
    h += dt * (alpha_h*(1.0 - h) - beta_h*h)
    n += dt * (alpha_n*(1.0 - n) - beta_n*n)

    # Membrane current calculations
    I_Na = g_Na * m^3 * h * (V - E_Na)
    I_K = g_K * n^4 * (V - E_K)
    I_L = g_L * (V - E_L)
    I_syn = g_syn * (V - E_syn)

    # Update membrane potential using forward Euler method
    V += dt * (1/C_m) * (-I_Na - I_K - I_L - I_syn) 

    # Store the results
    V_trace[i] = V
    I_syn_trace[i] = I_syn
end

# Plot V_trace on the first subplot and I_syn_trace on the second
add_trace!(plt_V, scatter(x=time, y=V_trace, mode="lines", name="V_m (g_syn_max = $(g_syn_max))"))
add_trace!(plt_I, scatter(x=time, y=I_syn_trace, mode="lines", name="I_syn (g_syn_max = $(g_syn_max))"))

# Display the plot with both subplots
[plt_V; plt_I]

## Finally, run simulation with different input strength

In [1]:
# Synaptic input parameters
t_start_syn = 10.0  # Synaptic input start time (ms)
t_end_syn = 12.0    # Synaptic input end time (ms)
E_syn = 0.0         # Synaptic reversal potential (mV)

# Different synaptic strengths to test
g_syn_max_values = [0, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0] # Maximum synaptic conductances (mS/cm^2)

# Create two plots
plt_V = PlotlyJS.plot()
plt_I = PlotlyJS.plot()

# Simulation loop for different synaptic strengths
for g_syn_max in g_syn_max_values
    # Initial conditions
    V = -65.0 # Initial membrane potential (mV)
    m = 0.05
    h = 0.6
    n = 0.32

    # Store the results for this synaptic strength
    V_trace = zeros(length(time))
    I_syn_trace = zeros(length(time))

    # Inner simulation loop (time steps)
    for i in 1:length(time)
        # Synaptic conductance (simple square pulse)
        g_syn = 0.0
        if t_start_syn <= time[i] <= t_end_syn
            g_syn = g_syn_max
        end

        # Alpha and beta functions for gating variables
        alpha_m = (0.1*(V + 40.0))/(1.0 - exp(-(V + 40.0)/10.0))
        beta_m = 4.0*exp(-(V + 65.0)/18.0)
        alpha_h = 0.07*exp(-(V + 65.0)/20.0)
        beta_h = 1.0/(1.0 + exp(-(V + 35.0)/10.0))
        alpha_n = (0.01*(V + 55.0))/(1.0 - exp(-(V + 55.0)/10.0))
        beta_n = 0.125*exp(-(V + 65.0)/80.0)

        # Update gating variables using forward Euler method
        m += dt * (alpha_m*(1.0 - m) - beta_m*m)
        h += dt * (alpha_h*(1.0 - h) - beta_h*h)
        n += dt * (alpha_n*(1.0 - n) - beta_n*n)

        # Membrane current calculations
        I_Na = g_Na * m^3 * h * (V - E_Na)
        I_K = g_K * n^4 * (V - E_K)
        I_L = g_L * (V - E_L)
        I_syn = g_syn * (V - E_syn)

        # Update membrane potential using forward Euler method
        V += dt * (1/C_m) * (-I_Na - I_K - I_L - I_syn) 

        # Store the results
        V_trace[i] = V
        I_syn_trace[i] = I_syn
    end

    # Plot V_trace on the first subplot and I_syn_trace on the second
    add_trace!(plt_V, scatter(x=time, y=V_trace, mode="lines", name="V_m (g_syn_max = $(g_syn_max))"))
    add_trace!(plt_I, scatter(x=time, y=I_syn_trace, mode="lines", name="I_syn (g_syn_max = $(g_syn_max))"))
end


# Display the plot with both subplots
[plt_V; plt_I]


UndefVarError: UndefVarError: `PlotlyJS` not defined

## Cable Theory

In [ ]:
using PlotlyJS

# --- Constants ---
# Membrane properties
C_m = 1.0   # Membrane capacitance (µF/cm^2)
g_L = 0.001 # Leak conductance (S/cm^2) 
E_L = -65.0  # Leak reversal potential (mV)

# Cable properties
d = 5.0     # Cable diameter (µm)
L = 1000.0  # Cable length (µm)
r_a = 100.0  # Axial resistivity (Ω*cm)

# Derived parameters
A = π * d^2 / 4  # Cross-sectional area (µm^2)
λ = sqrt(1 / (4 * r_a * g_L * A / 1e8)) # Space constant (µm)


In [ ]:

# --- Simulation parameters ---
dx = 20.0   # Spatial step (µm)
dt = 0.01  # Time step (ms)
t_end = 50.0 # Total simulation time (ms)
t = 0:dt:t_end

# --- Spatial discretization ---
x = 0:dx:L
Nx = length(x)

# --- Input current ---
I_inj = zeros(Nx)
I_inj[Int(round(Nx/2))] = 20.0 # Inject current at the middle
t_inj_start = 0
t_inj_end = 10 # ms

# --- Initialization ---
V = E_L * ones(Nx, length(t)) # Initialize membrane potential

# --- Simulation loop ---
for j in 2:length(t)
    # Calculate spatial derivatives (second-order central difference)
    d2Vdx2 = zeros(Nx)
    d2Vdx2[1] = (V[2, j-1] - 2*V[1, j-1] + V[1, j-1]) / dx^2  # Forward difference at x = 0
    d2Vdx2[2:Nx-1] = (V[1:Nx-2, j-1] - 2*V[2:Nx-1, j-1] + V[3:Nx, j-1]) / dx^2 # Central difference
    d2Vdx2[Nx] = (V[Nx-1, j-1] - 2*V[Nx, j-1] + V[Nx, j-1]) / dx^2  # Backward difference at x = L

    # Update membrane potential using the cable equation (with g_L)
    if t_inj_start <= t[j] <= t_inj_end
        V[:, j] = V[:, j-1] + dt/C_m * (d2Vdx2 ./ (4 * r_a * A / 1e8) .- g_L .* (V[:, j-1] .- E_L) .+ I_inj) 
    else
        V[:, j] = V[:, j-1] + dt/C_m * (d2Vdx2 ./ (4 * r_a * A / 1e8) .- g_L .* (V[:, j-1] .- E_L))
    end 
end


In [ ]:

# --- Plotting with PlotlyJS ---
traces = PlotlyJS.AbstractTrace[] 
for j in 1:100:length(t)
    trace = PlotlyJS.scatter(;x=x, y=V[:, j], mode="lines", name="t = $(round(t[j], digits=2)) ms") 
    push!(traces, trace)
end

layout = Layout(
    xaxis_title="Distance (µm)",
    yaxis_title="Membrane potential (mV)",
    title="Cable Equation Simulation (PlotlyJS)"
)

plt = PlotlyJS.plot(traces, layout)
display(plt)

In [ ]:
# --- Plotting with PlotlyJS (Time as x-axis) ---
traces = PlotlyJS.AbstractTrace[]
for i in 1:5:Int(round(Nx/2)) # Plot every 5th compartment
    trace = PlotlyJS.scatter(;x=t, y=V[i, :], mode="lines", name="x = $(x[i]) µm")
    push!(traces, trace)
end

layout = Layout(
    xaxis_title="Time (ms)",
    yaxis_title="Membrane potential (mV)",
    title="Cable Equation Simulation (PlotlyJS)"
)

plt = PlotlyJS.plot(traces, layout)
display(plt)